# TP2: Methodes de Newton et de Quasi-Newton a
## 02/06/2026
#### Léo SENTES Mia PERROUIN

### Exercice 1. Methode de Newton

#### 1.

In [22]:
import numpy as np

In [45]:
def methode_newton (g , Jacg , x0 , errmax ):
    iter=0
    x=np.array(x0)
    while np.linalg.norm(g(x))>=errmax:
        x = x + np.linalg.solve(Jacg(x),-g(x))
        iter+=1
    return x,iter

#### 2.
On cherche a tester notre code avec une fonction:

$f(x)=\begin{bmatrix} x_1^2 +x_2^2-2 \\x_1^2 - x_2^2-1\end{bmatrix}$


avec Jacg:
$Jacg(x)=\begin{bmatrix} 2x_1 & 2x_2 \\ 2x_1 & -2x_2\end{bmatrix}$


In [47]:
x0=np.array([2,4])
methode_newton(g,Jacg,x0,0.00001)

(array([1.22474487, 0.70710678]), 6)

In [50]:
x0=np.array([0,0])
methode_newton(g,Jacg,x0,0.00001)

LinAlgError: Singular matrix

Cela ne marche pas, c'est normal car la jacobienne n'est pas inversible.

In [57]:
def g3(x):
    return np.array([5*x[0]-np.cos(x[0]+x[1]),5*x[1]-np.sin(x[0]-x[1])])

def jacg3(x):
    x1 = x[0]
    x2 = x[1]
    
    # Ligne 1 de la matrice J
    row1 = [5 + np.sin(x1 + x2), np.sin(x1 + x2)]
    
    # Ligne 2 de la matrice J
    row2 = [-np.cos(x1 - x2), 5 + np.cos(x1 - x2)]
    
    # Assemblage de la matrice
    res = [row1, row2]
    
    return np.array(res)


In [58]:
x0=np.array([0,0])
methode_newton(g3,jacg3,x0,0.00001)

(array([0.19485942, 0.03235753]), 3)

Dans ce cas, le code aboutit car la jacobienne est inversible.

### 3.

In [59]:
def algo_newton (g ,h , x0 , errmax ):
    iter=0
    x=np.array(x0)
    while np.linalg.norm(g(x))>=errmax:
        x = x + np.linalg.solve(h(x),-g(x))
        iter+=1
    return x,iter

#### 4.
On teste notre fonction pour deux cas distincts:

$f_1(x) = \frac{(x_1-3)^4}{4}+\frac{7(x_2-5)^4}{4}$}


$f_2(x)=\frac{(x_1-3)^2}{2}+\frac{7(x_2-5)^2}{2}$

In [106]:
def f1(x):
    return (x[0]-3)**4 / 4 + 7*(x[1]-5)**4 / 4

def g1(x):
    return np.array([(x[0]-3)**3,7*(x[1]-5)**3])

def h1(x):
    return np.array([[3*(x[0]-3)**2,       0      ],
                     [0,21*(x[1]-5)**2]])
def f2(x):
    return (x[0]-3)**2 / 2 + 7*(x[1]-5)**2 / 2

def g2(x):
    return np.array([(x[0]-3),
                     7*(x[1]-5)])

def h2(x):
    return np.array([[1, 0],
                     [0, 7]])
x0=np.array([1,1])

In [107]:
algo_newton (g1 ,h1 , x0 , 0.00001 )

(array([2.99543268, 4.99086537]), 15)

In [108]:
algo_newton (g2 ,h2 , x0 , 0.00001 )

(array([3., 5.]), 1)

Dans les deux cas les codes convergent, nous remarquons que dans le second cas l'algorithme converge en 1 itération.
C'est dû au fait que la fonction 2 est d'ordre 2 donc son gradient est linéaire.

### Exercice 2. Methode de Quasi-Newton

#### 1.

In [109]:
def calcul_pas_wolfe (f ,g , xk , dk , eps1 , eps2 ):
    a=0
    b=1000
    sk=(a+b)/2
    iter=0
    for i in range (1000):
        if (f(xk+sk*dk)>(f(xk)+eps1*sk*np.dot(dk,g(xk)))):
            b=sk
        elif (np.dot(dk,g(xk+sk*dk))<(eps2*np.dot(dk,g(xk)))):
            a=sk
        else:
            break
        sk=(a+b)/2
        iter+=1
    return sk,iter
            

### 2.

In [110]:
def f1(x):
    return (x[0]-3)**4 / 4 + 7*(x[1]-5)**4 / 4
def g1(x):
    return np.array([(x[0]-3)**3,7*(x[1]-5)**3])
xk=np.array([1,1])    
calcul_pas_wolfe(f1,g1,xk,(-1)*g1(xk),0.0001,0.99)

(0.0152587890625, 15)

### 3.

In [116]:
def algo_quasi_newton_BFGS (f ,g , x0 , errmax ):
    I=len(x0)
    x1=x0
    B=np.eye(I)
    d=-np.dot(B,g(x1))
    s=calcul_pas_wolfe(f,g,x0,d,0.0001,0.99)
    while np.linalg.norm(g(x1))>errmax:
        d=-np.dot(B,g(x1))
        x2=x1+s*d
        sigma=x2-x1
        z=g(x2)-g(x1)
        A=np.eye(I)-((np.dot(sigma,z))/(np.dot(z,sigma)))
        B=np.dot(np.dot(A,B),A)+(np.dot(sigma,sigma)/np.dot(z,sigma))
        x1=x1-s*np.dot(B,g(x1))
    return x1
        

In [117]:
def f1(x):
    return (x[0]-3)**4 / 4 + 7*(x[1]-5)**4 / 4

def g1(x):
    return np.array([(x[0]-3)**3,7*(x[1]-5)**3])

def h1(x):
    return np.array([[3*(x[0]-3)**2,       0      ],
                     [0,21*(x[1]-5)**2]])
def f2(x):
    return (x[0]-3)**2 / 2 + 7*(x[1]-5)**2 / 2

def g2(x):
    return np.array([(x[0]-3),
                     7*(x[1]-5)])

def h2(x):
    return np.array([[1, 0],
                     [0, 7]])
x0=np.array([1,1])

In [118]:
algo_quasi_newton_BFGS (f2 ,g2 , x0 , 0.00001 )

C:\Users\miaperro\AppData\Local\Temp\ipykernel_13556\3134781849.py:12: RuntimeWarning: invalid value encountered in scalar divide
  A=np.eye(I)-((np.dot(sigma,z))/(np.dot(z,sigma)))


array([nan, nan])